# Experiment: Simple Two-Sideband Processing Example

Minimal app-facing workflow using `processing.py`:

- choose a dataset folder
- choose a passage and harmonic
- optionally override the detected filter vertical center and width
- load the cached stack with `load_passage(...)`
- extract display-ready views with `get_view_image(...)`
- inspect the current processing settings and diagnostics


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
assert (repo_root / "processing.py").exists(), "Start Jupyter from the repository root."
sys.path.insert(0, str(repo_root))

from processing import get_view_image, load_passage

plt.rcParams["figure.figsize"] = (15, 10)
np.set_printoptions(precision=4, suppress=True)

print(f"Using repo root: {repo_root}")


In [ ]:
folder_path = ""
passage = "forward"
harmonic_index = 4

processing_mode = "two_sideband"
pad_fact = 4
alpha = 0.3
carrier_row_override = None
filter_width_override = None

assert folder_path.strip(), "Set folder_path to your dataset directory before running this cell."
assert passage in {"forward", "reverse"}, "passage must be 'forward' or 'reverse'"
assert 0 <= harmonic_index <= 5, "harmonic_index must be between 0 and 5"


In [ ]:
loaded = load_passage(
    folder_path,
    passage=passage,
    processing_mode=processing_mode,
    pad_fact=pad_fact,
    alpha=alpha,
    carrier_row_override=carrier_row_override,
    filter_width_override=filter_width_override,
)

raw_amplitude = get_view_image(loaded.stage_stacks, harmonic_index, "raw", "amplitude")
raw_phase = get_view_image(loaded.stage_stacks, harmonic_index, "raw", "phase")
processed_amplitude = get_view_image(
    loaded.stage_stacks,
    harmonic_index,
    "processed",
    "amplitude",
    processing_settings=loaded.processing_settings,
)
processed_phase = get_view_image(
    loaded.stage_stacks,
    harmonic_index,
    "processed",
    "phase",
    processing_settings=loaded.processing_settings,
)
mag_signal_ft = get_view_image(loaded.stage_stacks, harmonic_index, "mag_signal_ft", "amplitude")
filtered_shift = get_view_image(loaded.stage_stacks, harmonic_index, "filtered_shift", "amplitude")

summary = {
    "folder_path": loaded.folder_path,
    "cache_path": loaded.cache_path,
    "image_name": loaded.image_name,
    "passage": loaded.passage,
    "harmonic_index": harmonic_index,
    "shape": loaded.o[:, :, harmonic_index].shape,
    "processing_mode": loaded.processing_settings["processing_mode"],
    "carrier_row_override": carrier_row_override,
    "filter_width_override": filter_width_override,
    "auto_center_row": loaded.processing_settings["auto_center_row"],
    "current_center_row": loaded.processing_settings["current_center_row"],
    "auto_filter_width_y": loaded.processing_settings["auto_filter_width_y"],
    "current_filter_width_y": loaded.processing_settings["current_filter_width_y"],
    "rotation_angle_deg": loaded.processing_diagnostics["rotation_angle_deg_by_harmonic"][harmonic_index],
    "mirror_row": loaded.processing_diagnostics["mirror_row_by_harmonic"][harmonic_index],
}
summary


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
plots = [
    (raw_amplitude, "Raw amplitude", "hot"),
    (processed_amplitude, "Processed amplitude", "hot"),
    (np.log1p(mag_signal_ft), "Log FFT magnitude", "magma"),
    (raw_phase, "Raw phase", "twilight"),
    (processed_phase, "Processed phase", "bwr"),
    (np.log1p(filtered_shift), "Log filtered shift magnitude", "magma"),
]

for axis, (image, title, cmap) in zip(axes.ravel(), plots):
    im = axis.imshow(image, aspect="auto", cmap=cmap)
    axis.set_title(title)
    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    fig.colorbar(im, ax=axis, fraction=0.046, pad=0.04)

plt.show()


In [ ]:
view_shapes = {
    "raw_amplitude": raw_amplitude.shape,
    "raw_phase": raw_phase.shape,
    "processed_amplitude": processed_amplitude.shape,
    "processed_phase": processed_phase.shape,
    "mag_signal_ft": mag_signal_ft.shape,
    "filtered_shift": filtered_shift.shape,
}
view_shapes
